# Tiled Multi-GPU Deconvolution

Interactive front-end for tuning deconvolution parameters and inspecting the
result in BigDataViewer. The heavy lifting is done by the BIOP /
BigDataViewer-Playground commands driven through PyImageJ.

**Requirements:** an OpenCL-capable GPU, and `uv sync` already run at the repo
root. Start Jupyter with `uv run jupyter lab`. A JDK and Maven are downloaded
automatically on first use (via cjdk) if not already present -- no conda needed.

> Set the JVM mode **before** the first `init_imagej()` call -- a JVM starts
> only once per kernel. Use `interactive` here so BigDataViewer windows can open.

In [ ]:
from deconvolve import DeconvolveParams, init_imagej, run

# 'interactive' allows BigDataViewer windows; use 'headless' for save-only.
ij = init_imagej(mode="interactive", max_heap="32g")

In [ ]:
params = DeconvolveParams(
    image_file=r"/path/to/image.czi",        # 5D: XYZ + channels + timepoints
    psf_file=r"/path/to/psf.tif",            # single channel, reused for all channels
    output_folder=r"/path/to/output_folder",
    num_iterations=120,
    regularization_factor=0.0,               # raise to tame noise / ringing
    block_size_z=64,                         # lower if you run out of GPU memory
    n_threads=10,                            # CPU workers feeding the GPU pool
    show_in_bdv=True,   # open raw + deconvolved in BigDataViewer
    save_output=True,   # also write the OME-TIFF
    overwrite=False,
)
params

In [ ]:
out_path = run(params, ij=ij)
print("Output:", out_path)

The computation is **lazy**: browsing the sources in BigDataViewer or writing
the OME-TIFF is what actually triggers the block-by-block GPU work.

Tune parameters (iterations, regularization, block size) by editing the
`DeconvolveParams` cell above and re-running the last cell -- no kernel restart
needed.